In [1]:
# ============================================================
# 03_AURORA_model_zoo_baselines.ipynb
# AURORA-TWETF Model Zoo Baselines
#
# Purpose:
# 1. Load the leakage-controlled modeling dataset from Notebook 02.
# 2. Define chronological train / validation / test splits.
# 3. Train baseline multiclass regime classifiers for 20d and 60d targets.
# 4. Evaluate with accuracy, balanced accuracy, macro-F1, weighted-F1,
#    quadratic weighted kappa, ordinal MAE, ordinal RMSE, and class-wise metrics.
# 5. Save predictions, probabilities, metrics, confusion matrices,
#    feature importances, plots, and model artifacts.
#
# This notebook establishes baseline M0/M1/M2/M3 models.
# Advanced AURORA-QC uncertainty-aware ordinal allocation starts later.
# ============================================================

from __future__ import annotations

import os
import sys
import json
import math
import time
import random
import pickle
import hashlib
import subprocess
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name
    try:
        return __import__(import_name)
    except Exception:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        return __import__(import_name)

pd = install_if_missing("pandas", "pandas")
np = install_if_missing("numpy", "numpy")
sklearn = install_if_missing("scikit-learn", "sklearn")
matplotlib = install_if_missing("matplotlib", "matplotlib")
seaborn = install_if_missing("seaborn", "seaborn")
joblib = install_if_missing("joblib", "joblib")
pyarrow = install_if_missing("pyarrow", "pyarrow")

try:
    lightgbm = install_if_missing("lightgbm", "lightgbm")
    HAS_LIGHTGBM = True
except Exception as e:
    print("LightGBM unavailable. Skipping LightGBM models.")
    print(e)
    HAS_LIGHTGBM = False

try:
    xgboost = install_if_missing("xgboost", "xgboost")
    HAS_XGBOOST = True
except Exception as e:
    print("XGBoost unavailable. Skipping XGBoost models.")
    print(e)
    HAS_XGBOOST = False

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    cohen_kappa_score,
    log_loss,
)
from sklearn.utils.class_weight import compute_class_weight

if HAS_LIGHTGBM:
    from lightgbm import LGBMClassifier

if HAS_XGBOOST:
    from xgboost import XGBClassifier

# ============================================================
# 1. Reproducibility and paths
# ============================================================

RANDOM_SEED = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

PROJECT_CODE = "AURORA_TWETF"

PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
MODEL_ZOO_ROOT = OUTPUT_ROOT / "model_zoo_baselines"
RUN_ROOT = MODEL_ZOO_ROOT / f"run_{RUN_ID}"

TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"
MODEL_DIR = OUTPUT_ROOT / "models"

PRED_DIR = RUN_ROOT / "predictions"
PROBA_DIR = RUN_ROOT / "probabilities"
METRIC_DIR = RUN_ROOT / "metrics"
PLOT_DIR = RUN_ROOT / "plots"
ARTIFACT_MODEL_DIR = RUN_ROOT / "models"
IMPORTANCE_DIR = RUN_ROOT / "feature_importance"
SPLIT_DIR = RUN_ROOT / "splits"

for d in [
    DATA_ROOT,
    MODELING_DIR,
    OUTPUT_ROOT,
    MODEL_ZOO_ROOT,
    RUN_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    MODEL_DIR,
    PRED_DIR,
    PROBA_DIR,
    METRIC_DIR,
    PLOT_DIR,
    ARTIFACT_MODEL_DIR,
    IMPORTANCE_DIR,
    SPLIT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_DATA_PATH = MODELING_DIR / "AURORA_TWETF_features_with_labels.parquet"

print("=" * 80)
print("AURORA-TWETF Model Zoo Baselines")
print("=" * 80)
print("Timestamp UTC:", RUN_TIMESTAMP)
print("Run ID       :", RUN_ID)
print("Project root :", PUBLICATION_ROOT)
print("Input data   :", MODEL_DATA_PATH)
print("Run root     :", RUN_ROOT)
print("=" * 80)

# ============================================================
# 2. Configuration
# ============================================================

TARGET_COLS = [
    "TAIEX_regime_fixed_20d",
    "TAIEX_regime_fixed_60d",
]

CLASS_LABELS = [0, 1, 2, 3, 4]

REGIME_LABEL_DEFINITION = {
    0: "Strong Bear",
    1: "Bear",
    2: "Neutral",
    3: "Bull",
    4: "Strong Bull",
}

# Chronological split.
# For 1262 rows, this gives approximately:
# train 70%, validation 15%, test 15%.
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9

# To avoid extremely slow experiments in Colab, keep baseline models moderate.
N_JOBS = -1

# The model zoo.
# M0 = non-learning baselines.
# M1+ = learning baselines.
MODEL_CONFIG = {
    "M0_dummy_most_frequent": {
        "type": "dummy_most_frequent",
        "enabled": True,
    },
    "M0_dummy_stratified": {
        "type": "dummy_stratified",
        "enabled": True,
    },
    "M0_persistence_previous_label": {
        "type": "persistence",
        "enabled": True,
    },
    "M1_logistic_regression_balanced": {
        "type": "logistic_regression",
        "enabled": True,
    },
    "M2_random_forest_balanced": {
        "type": "random_forest",
        "enabled": True,
    },
    "M3_extra_trees_balanced": {
        "type": "extra_trees",
        "enabled": True,
    },
    "M4_hist_gradient_boosting": {
        "type": "hist_gradient_boosting",
        "enabled": True,
    },
    "M5_lightgbm_balanced": {
        "type": "lightgbm",
        "enabled": HAS_LIGHTGBM,
    },
    "M6_xgboost_multiclass": {
        "type": "xgboost",
        "enabled": HAS_XGBOOST,
    },
}

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    path = Path(path)
    path.write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return str(x).replace("/", "_").replace("\\", "_").replace(":", "_").replace(" ", "_")

def ordinal_mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

def ordinal_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def adjacent_accuracy(y_true, y_pred, tolerance=1):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    return float(np.mean(np.abs(y_true - y_pred) <= tolerance))

def expected_class_from_proba(proba, labels=CLASS_LABELS):
    labels_arr = np.asarray(labels, dtype=float)
    return proba @ labels_arr

def proba_to_pred(proba, labels=CLASS_LABELS):
    labels_arr = np.asarray(labels, dtype=int)
    return labels_arr[np.argmax(proba, axis=1)]

def align_proba_columns(proba, model_classes, all_classes=CLASS_LABELS):
    """
    Ensures probability matrix has exactly one column per class in all_classes.
    Some models may omit a class if it was absent in training.
    """
    proba = np.asarray(proba, dtype=float)
    out = np.zeros((proba.shape[0], len(all_classes)), dtype=float)

    class_to_pos = {int(c): i for i, c in enumerate(model_classes)}
    for j, c in enumerate(all_classes):
        if int(c) in class_to_pos:
            out[:, j] = proba[:, class_to_pos[int(c)]]

    row_sums = out.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    out = out / row_sums
    return out

def evaluate_predictions(y_true, y_pred, proba=None, labels=CLASS_LABELS):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    metrics = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "quadratic_weighted_kappa": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
        "ordinal_mae": ordinal_mae(y_true, y_pred),
        "ordinal_rmse": ordinal_rmse(y_true, y_pred),
        "adjacent_accuracy_tol_1": adjacent_accuracy(y_true, y_pred, tolerance=1),
        "large_error_rate_abs_ge_2": float(np.mean(np.abs(y_true - y_pred) >= 2)),
        "extreme_error_rate_abs_ge_3": float(np.mean(np.abs(y_true - y_pred) >= 3)),
    }

    if proba is not None:
        try:
            metrics["multiclass_log_loss"] = float(log_loss(y_true, proba, labels=labels))
        except Exception:
            metrics["multiclass_log_loss"] = np.nan

        try:
            exp_class = expected_class_from_proba(proba, labels=labels)
            metrics["expected_class_mae"] = float(np.mean(np.abs(y_true - exp_class)))
            metrics["expected_class_rmse"] = float(np.sqrt(np.mean((y_true - exp_class) ** 2)))
        except Exception:
            metrics["expected_class_mae"] = np.nan
            metrics["expected_class_rmse"] = np.nan
    else:
        metrics["multiclass_log_loss"] = np.nan
        metrics["expected_class_mae"] = np.nan
        metrics["expected_class_rmse"] = np.nan

    return metrics

def make_classification_report_df(y_true, y_pred, labels=CLASS_LABELS):
    target_names = [REGIME_LABEL_DEFINITION[c] for c in labels]
    rep = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=target_names,
        output_dict=True,
        zero_division=0,
    )
    return pd.DataFrame(rep).T.reset_index().rename(columns={"index": "class_or_average"})

def plot_confusion_matrix(cm, title, path, labels=CLASS_LABELS, normalize=False):
    plt.figure(figsize=(7.5, 6))
    if normalize:
        cm_plot = cm.astype(float)
        row_sums = cm_plot.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        cm_plot = cm_plot / row_sums
        fmt = ".2f"
    else:
        cm_plot = cm
        fmt = "d"

    tick_labels = [f"{c}\n{REGIME_LABEL_DEFINITION[c]}" for c in labels]

    sns.heatmap(
        cm_plot,
        annot=True,
        fmt=fmt,
        cmap="Blues",
        xticklabels=tick_labels,
        yticklabels=tick_labels,
        cbar=True,
    )
    plt.title(title)
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def plot_class_distribution(y, title, path, labels=CLASS_LABELS):
    counts = pd.Series(y).value_counts().reindex(labels, fill_value=0)
    shares = counts / counts.sum()

    plt.figure(figsize=(8, 4.5))
    ax = sns.barplot(x=[str(c) for c in labels], y=shares.values, color="#4C72B0")
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Share")
    plt.ylim(0, max(0.50, shares.max() * 1.20))

    for i, v in enumerate(shares.values):
        ax.text(i, v + 0.01, f"{v:.1%}\n(n={counts.iloc[i]})", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def plot_metric_comparison(metric_df, target_col, split_name, metric_name, path):
    df = metric_df[
        (metric_df["target_col"] == target_col)
        & (metric_df["split"] == split_name)
    ].copy()

    df = df.sort_values(metric_name, ascending=False)

    plt.figure(figsize=(10, max(4, 0.45 * len(df))))
    sns.barplot(data=df, y="model_name", x=metric_name, color="#55A868")
    plt.title(f"{target_col} | {split_name} | {metric_name}")
    plt.xlabel(metric_name)
    plt.ylabel("Model")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def plot_feature_importance(fi_df, title, path, top_n=30):
    if fi_df is None or fi_df.empty:
        return

    df = fi_df.sort_values("importance", ascending=False).head(top_n).copy()

    plt.figure(figsize=(10, max(5, 0.32 * len(df))))
    sns.barplot(data=df, x="importance", y="feature", color="#C44E52")
    plt.title(title)
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

# ============================================================
# 4. Persistence baseline
# ============================================================

class PreviousLabelPersistenceClassifier(BaseEstimator, ClassifierMixin):
    """
    A baseline classifier for chronological datasets.

    It predicts the most recently observed label from the immediately previous row.
    For the first row in a split, it uses fallback_class.

    This is not a fitted ML model. It is a regime-persistence benchmark.
    """

    def __init__(self, fallback_class=2, labels=CLASS_LABELS):
        self.fallback_class = int(fallback_class)
        self.labels = list(labels)
        self.classes_ = np.asarray(labels, dtype=int)

    def fit(self, X, y):
        y = np.asarray(y, dtype=int)
        self.train_last_class_ = int(y[-1]) if len(y) else self.fallback_class
        counts = pd.Series(y).value_counts().reindex(self.labels, fill_value=0)
        if counts.sum() > 0:
            self.class_prior_ = (counts / counts.sum()).values
        else:
            self.class_prior_ = np.ones(len(self.labels)) / len(self.labels)
        return self

    def predict_with_previous_series(self, y_previous_available):
        """
        y_previous_available should be a series indexed like the evaluation split,
        containing the previous chronological label for each evaluation date.
        """
        prev = pd.Series(y_previous_available).copy()
        pred = prev.fillna(self.train_last_class_).astype(int).values
        return pred

    def predict(self, X):
        # Fallback behavior if no chronological previous label is supplied.
        return np.repeat(self.fallback_class, len(X)).astype(int)

    def predict_proba_from_pred(self, y_pred):
        proba = np.zeros((len(y_pred), len(self.labels)), dtype=float)
        for i, pred in enumerate(y_pred):
            if pred in self.labels:
                proba[i, self.labels.index(int(pred))] = 1.0
            else:
                proba[i, self.labels.index(self.fallback_class)] = 1.0
        return proba

# ============================================================
# 5. Load modeling dataset
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading modeling dataset")
print("=" * 80)

if not MODEL_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing modeling dataset: {MODEL_DATA_PATH}\n"
        "Please run 02_AURORA_data_preparation_leakage_controlled.ipynb first."
    )

df = pd.read_parquet(MODEL_DATA_PATH)
df.index = pd.to_datetime(df.index)
df = df.sort_index()

print("Dataset shape:", df.shape)
print("Date range   :", df.index.min().date(), "to", df.index.max().date())
print("Columns      :", len(df.columns))

missing_targets = [c for c in TARGET_COLS if c not in df.columns]
if missing_targets:
    raise ValueError(f"Missing target columns: {missing_targets}")

# Ensure target columns are integer.
for target_col in TARGET_COLS:
    df[target_col] = df[target_col].astype(int)

# Candidate feature columns.
# Exclude target columns and any obvious audit/future columns if present.
EXCLUDE_KEYWORDS = [
    "future",
    "audit",
    "target",
    "regime_fixed",
]

feature_cols = []
for c in df.columns:
    if c in TARGET_COLS:
        continue
    c_lower = str(c).lower()
    if any(k in c_lower for k in EXCLUDE_KEYWORDS):
        continue
    feature_cols.append(c)

print("Feature columns:", len(feature_cols))
print("Target columns :", TARGET_COLS)

# Save dataset summary.
dataset_summary = {
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "input_path": str(MODEL_DATA_PATH),
    "dataset_shape": df.shape,
    "date_start": str(df.index.min().date()),
    "date_end": str(df.index.max().date()),
    "n_features": len(feature_cols),
    "target_cols": TARGET_COLS,
    "feature_cols": feature_cols,
}

save_json(REPORT_DIR / f"AURORA_03_dataset_summary_{RUN_ID}.json", dataset_summary)
save_json(RUN_ROOT / "dataset_summary.json", dataset_summary)

# ============================================================
# 6. Chronological train / validation / test split
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Chronological train / validation / test split")
print("=" * 80)

n = len(df)
train_end = int(np.floor(n * TRAIN_FRAC))
val_end = int(np.floor(n * (TRAIN_FRAC + VAL_FRAC)))

train_idx = df.index[:train_end]
val_idx = df.index[train_end:val_end]
test_idx = df.index[val_end:]

split_df = pd.DataFrame(index=df.index)
split_df["split"] = "unused"
split_df.loc[train_idx, "split"] = "train"
split_df.loc[val_idx, "split"] = "validation"
split_df.loc[test_idx, "split"] = "test"

split_report = pd.DataFrame([
    {
        "split": "train",
        "n": len(train_idx),
        "start_date": train_idx.min().date(),
        "end_date": train_idx.max().date(),
    },
    {
        "split": "validation",
        "n": len(val_idx),
        "start_date": val_idx.min().date(),
        "end_date": val_idx.max().date(),
    },
    {
        "split": "test",
        "n": len(test_idx),
        "start_date": test_idx.min().date(),
        "end_date": test_idx.max().date(),
    },
])

print(split_report.to_string(index=False))

split_report.to_csv(TABLE_DIR / f"table_10_model_zoo_split_report_{RUN_ID}.csv", index=False)
split_report.to_csv(RUN_ROOT / "split_report.csv", index=False)
split_df.to_csv(SPLIT_DIR / "row_split_assignment.csv")

# Save split indices.
save_json(SPLIT_DIR / "split_indices.json", {
    "train_start": str(train_idx.min()),
    "train_end": str(train_idx.max()),
    "validation_start": str(val_idx.min()),
    "validation_end": str(val_idx.max()),
    "test_start": str(test_idx.min()),
    "test_end": str(test_idx.max()),
    "n_train": len(train_idx),
    "n_validation": len(val_idx),
    "n_test": len(test_idx),
})

# Plot target distributions per split.
for target_col in TARGET_COLS:
    for split_name, idx in [
        ("train", train_idx),
        ("validation", val_idx),
        ("test", test_idx),
    ]:
        plot_class_distribution(
            df.loc[idx, target_col],
            title=f"{target_col} class distribution | {split_name}",
            path=PLOT_DIR / f"class_distribution_{safe_name(target_col)}_{split_name}.png",
        )

# ============================================================
# 7. Model factory
# ============================================================

def build_model(model_type, y_train=None):
    """
    Build a model pipeline.
    """
    if model_type == "dummy_most_frequent":
        return DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)

    if model_type == "dummy_stratified":
        return DummyClassifier(strategy="stratified", random_state=RANDOM_SEED)

    if model_type == "persistence":
        if y_train is not None and len(y_train) > 0:
            fallback_class = int(pd.Series(y_train).mode().iloc[0])
        else:
            fallback_class = 2
        return PreviousLabelPersistenceClassifier(fallback_class=fallback_class)

    if model_type == "logistic_regression":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                C=1.0,
                solver="lbfgs",
                multi_class="auto",
                class_weight="balanced",
                max_iter=3000,
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
            )),
        ])

    if model_type == "random_forest":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=10,
                max_features="sqrt",
                class_weight="balanced_subsample",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
            )),
        ])

    if model_type == "extra_trees":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", ExtraTreesClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=10,
                max_features="sqrt",
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
            )),
        ])

    if model_type == "hist_gradient_boosting":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", HistGradientBoostingClassifier(
                loss="log_loss",
                learning_rate=0.04,
                max_iter=250,
                max_leaf_nodes=15,
                l2_regularization=0.10,
                early_stopping=True,
                validation_fraction=0.15,
                random_state=RANDOM_SEED,
            )),
        ])

    if model_type == "lightgbm":
        if not HAS_LIGHTGBM:
            raise RuntimeError("LightGBM is not available.")

        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", LGBMClassifier(
                objective="multiclass",
                num_class=len(CLASS_LABELS),
                n_estimators=500,
                learning_rate=0.03,
                num_leaves=15,
                max_depth=5,
                min_child_samples=20,
                subsample=0.80,
                colsample_bytree=0.80,
                reg_alpha=0.10,
                reg_lambda=0.30,
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
                verbose=-1,
            )),
        ])

    if model_type == "xgboost":
        if not HAS_XGBOOST:
            raise RuntimeError("XGBoost is not available.")

        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                objective="multi:softprob",
                num_class=len(CLASS_LABELS),
                n_estimators=400,
                learning_rate=0.03,
                max_depth=3,
                min_child_weight=3,
                subsample=0.80,
                colsample_bytree=0.80,
                reg_alpha=0.10,
                reg_lambda=1.00,
                eval_metric="mlogloss",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
                verbosity=0,
            )),
        ])

    raise ValueError(f"Unknown model_type: {model_type}")

def get_model_classes(model):
    """
    Get classes_ from model or final pipeline estimator.
    """
    if hasattr(model, "classes_"):
        return model.classes_
    if isinstance(model, Pipeline):
        clf = model.named_steps.get("clf")
        if hasattr(clf, "classes_"):
            return clf.classes_
    return np.asarray(CLASS_LABELS, dtype=int)

def get_feature_importance(model, feature_cols):
    """
    Extract feature importances or coefficients when available.
    """
    clf = model
    if isinstance(model, Pipeline):
        clf = model.named_steps.get("clf", model)

    if hasattr(clf, "feature_importances_"):
        imp = np.asarray(clf.feature_importances_, dtype=float)
        return pd.DataFrame({
            "feature": feature_cols,
            "importance": imp,
        }).sort_values("importance", ascending=False)

    if hasattr(clf, "coef_"):
        coef = np.asarray(clf.coef_, dtype=float)
        if coef.ndim == 2:
            imp = np.mean(np.abs(coef), axis=0)
        else:
            imp = np.abs(coef)
        return pd.DataFrame({
            "feature": feature_cols,
            "importance": imp,
        }).sort_values("importance", ascending=False)

    return pd.DataFrame(columns=["feature", "importance"])

# ============================================================
# 8. Training and evaluation loop
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Training and evaluating baseline model zoo")
print("=" * 80)

all_metric_rows = []
all_class_report_rows = []
all_prediction_frames = []
all_probability_frames = []
all_feature_importance_frames = []
run_model_summaries = []

X_all = df[feature_cols].copy()

for target_col in TARGET_COLS:
    print("\n" + "-" * 80)
    print(f"Target: {target_col}")
    print("-" * 80)

    y_all = df[target_col].astype(int).copy()

    X_train = X_all.loc[train_idx]
    X_val = X_all.loc[val_idx]
    X_test = X_all.loc[test_idx]

    y_train = y_all.loc[train_idx]
    y_val = y_all.loc[val_idx]
    y_test = y_all.loc[test_idx]

    # Previous labels for persistence baseline.
    y_prev_all = y_all.shift(1)

    # Save target split distribution.
    target_split_distribution_rows = []
    for split_name, idx in [
        ("train", train_idx),
        ("validation", val_idx),
        ("test", test_idx),
    ]:
        counts = y_all.loc[idx].value_counts().reindex(CLASS_LABELS, fill_value=0)
        shares = counts / counts.sum()
        for c in CLASS_LABELS:
            target_split_distribution_rows.append({
                "target_col": target_col,
                "split": split_name,
                "class": c,
                "regime_name": REGIME_LABEL_DEFINITION[c],
                "n": int(counts.loc[c]),
                "share": float(shares.loc[c]),
            })

    target_split_distribution = pd.DataFrame(target_split_distribution_rows)
    target_split_distribution.to_csv(
        TABLE_DIR / f"table_11_target_split_distribution_{safe_name(target_col)}_{RUN_ID}.csv",
        index=False,
    )
    target_split_distribution.to_csv(
        RUN_ROOT / f"target_split_distribution_{safe_name(target_col)}.csv",
        index=False,
    )

    for model_name, cfg in MODEL_CONFIG.items():
        if not cfg.get("enabled", True):
            print(f"Skipping disabled model: {model_name}")
            continue

        model_type = cfg["type"]

        print(f"\nTraining model: {model_name}")

        start_time = time.time()
        model = build_model(model_type, y_train=y_train)

        try:
            if model_type == "persistence":
                model.fit(X_train, y_train)
            else:
                model.fit(X_train, y_train)

            fit_seconds = time.time() - start_time

            split_objects = {
                "train": (X_train, y_train, train_idx),
                "validation": (X_val, y_val, val_idx),
                "test": (X_test, y_test, test_idx),
            }

            model_prediction_frames = []
            model_probability_frames = []

            for split_name, (X_split, y_split, idx_split) in split_objects.items():
                if model_type == "persistence":
                    y_pred = model.predict_with_previous_series(y_prev_all.loc[idx_split])
                    proba = model.predict_proba_from_pred(y_pred)
                else:
                    y_pred = model.predict(X_split).astype(int)

                    if hasattr(model, "predict_proba"):
                        raw_proba = model.predict_proba(X_split)
                        model_classes = get_model_classes(model)
                        proba = align_proba_columns(raw_proba, model_classes, CLASS_LABELS)
                    else:
                        proba = None

                metric_values = evaluate_predictions(
                    y_true=y_split.values,
                    y_pred=y_pred,
                    proba=proba,
                    labels=CLASS_LABELS,
                )

                metric_row = {
                    "run_id": RUN_ID,
                    "run_timestamp_utc": RUN_TIMESTAMP,
                    "target_col": target_col,
                    "model_name": model_name,
                    "model_type": model_type,
                    "split": split_name,
                    "fit_seconds": float(fit_seconds),
                    **metric_values,
                }
                all_metric_rows.append(metric_row)

                print(
                    f"{split_name:>10} | "
                    f"acc={metric_values['accuracy']:.4f} | "
                    f"bal_acc={metric_values['balanced_accuracy']:.4f} | "
                    f"macro_f1={metric_values['macro_f1']:.4f} | "
                    f"ord_mae={metric_values['ordinal_mae']:.4f} | "
                    f"qwk={metric_values['quadratic_weighted_kappa']:.4f}"
                )

                # Class report.
                cr_df = make_classification_report_df(
                    y_true=y_split.values,
                    y_pred=y_pred,
                    labels=CLASS_LABELS,
                )
                cr_df.insert(0, "split", split_name)
                cr_df.insert(0, "model_name", model_name)
                cr_df.insert(0, "target_col", target_col)
                cr_df.insert(0, "run_id", RUN_ID)
                all_class_report_rows.append(cr_df)

                # Confusion matrix.
                cm = confusion_matrix(y_split.values, y_pred, labels=CLASS_LABELS)
                cm_df = pd.DataFrame(
                    cm,
                    index=[f"true_{c}" for c in CLASS_LABELS],
                    columns=[f"pred_{c}" for c in CLASS_LABELS],
                )
                cm_path = METRIC_DIR / f"confusion_matrix_{safe_name(target_col)}_{model_name}_{split_name}.csv"
                cm_df.to_csv(cm_path)

                plot_confusion_matrix(
                    cm,
                    title=f"{target_col} | {model_name} | {split_name}",
                    path=PLOT_DIR / f"confusion_matrix_{safe_name(target_col)}_{model_name}_{split_name}.png",
                    labels=CLASS_LABELS,
                    normalize=False,
                )

                plot_confusion_matrix(
                    cm,
                    title=f"{target_col} | {model_name} | {split_name} | normalized",
                    path=PLOT_DIR / f"confusion_matrix_normalized_{safe_name(target_col)}_{model_name}_{split_name}.png",
                    labels=CLASS_LABELS,
                    normalize=True,
                )

                # Prediction frame.
                pred_df = pd.DataFrame(index=idx_split)
                pred_df.index.name = "date"
                pred_df["run_id"] = RUN_ID
                pred_df["target_col"] = target_col
                pred_df["model_name"] = model_name
                pred_df["model_type"] = model_type
                pred_df["split"] = split_name
                pred_df["y_true"] = y_split.values.astype(int)
                pred_df["y_pred"] = y_pred.astype(int)
                pred_df["abs_class_error"] = np.abs(pred_df["y_true"] - pred_df["y_pred"])
                pred_df["squared_class_error"] = (pred_df["y_true"] - pred_df["y_pred"]) ** 2

                if proba is not None:
                    exp_class = expected_class_from_proba(proba, labels=CLASS_LABELS)
                    pred_df["expected_class"] = exp_class
                    pred_df["expected_class_error"] = pred_df["y_true"] - pred_df["expected_class"]
                    pred_df["max_probability"] = np.max(proba, axis=1)
                    pred_df["pred_entropy"] = -np.sum(
                        np.where(proba > 0, proba * np.log(np.clip(proba, 1e-15, 1.0)), 0.0),
                        axis=1,
                    )
                else:
                    pred_df["expected_class"] = np.nan
                    pred_df["expected_class_error"] = np.nan
                    pred_df["max_probability"] = np.nan
                    pred_df["pred_entropy"] = np.nan

                model_prediction_frames.append(pred_df)
                all_prediction_frames.append(pred_df)

                # Probability frame.
                if proba is not None:
                    proba_df = pd.DataFrame(
                        proba,
                        index=idx_split,
                        columns=[f"proba_class_{c}" for c in CLASS_LABELS],
                    )
                    proba_df.index.name = "date"
                    proba_df.insert(0, "split", split_name)
                    proba_df.insert(0, "model_type", model_type)
                    proba_df.insert(0, "model_name", model_name)
                    proba_df.insert(0, "target_col", target_col)
                    proba_df.insert(0, "run_id", RUN_ID)

                    model_probability_frames.append(proba_df)
                    all_probability_frames.append(proba_df)

            # Save per-model predictions and probabilities.
            if model_prediction_frames:
                model_pred_all = pd.concat(model_prediction_frames, axis=0)
                model_pred_all.to_parquet(
                    PRED_DIR / f"predictions_{safe_name(target_col)}_{model_name}.parquet"
                )
                model_pred_all.to_csv(
                    PRED_DIR / f"predictions_{safe_name(target_col)}_{model_name}.csv"
                )

            if model_probability_frames:
                model_proba_all = pd.concat(model_probability_frames, axis=0)
                model_proba_all.to_parquet(
                    PROBA_DIR / f"probabilities_{safe_name(target_col)}_{model_name}.parquet"
                )
                model_proba_all.to_csv(
                    PROBA_DIR / f"probabilities_{safe_name(target_col)}_{model_name}.csv"
                )

            # Feature importance.
            fi_df = get_feature_importance(model, feature_cols)
            if fi_df is not None and not fi_df.empty:
                fi_df.insert(0, "model_type", model_type)
                fi_df.insert(0, "model_name", model_name)
                fi_df.insert(0, "target_col", target_col)
                fi_df.insert(0, "run_id", RUN_ID)

                fi_path = IMPORTANCE_DIR / f"feature_importance_{safe_name(target_col)}_{model_name}.csv"
                fi_df.to_csv(fi_path, index=False)

                plot_feature_importance(
                    fi_df,
                    title=f"{target_col} | {model_name} | top feature importances",
                    path=PLOT_DIR / f"feature_importance_{safe_name(target_col)}_{model_name}.png",
                    top_n=30,
                )

                all_feature_importance_frames.append(fi_df)

            # Save fitted model.
            model_path = ARTIFACT_MODEL_DIR / f"model_{safe_name(target_col)}_{model_name}.joblib"
            joblib.dump(model, model_path)

            run_model_summaries.append({
                "target_col": target_col,
                "model_name": model_name,
                "model_type": model_type,
                "status": "success",
                "fit_seconds": float(fit_seconds),
                "model_path": str(model_path),
            })

        except Exception as e:
            print(f"ERROR in model {model_name} for target {target_col}: {repr(e)}")
            run_model_summaries.append({
                "target_col": target_col,
                "model_name": model_name,
                "model_type": model_type,
                "status": "failed",
                "error": repr(e),
            })

# ============================================================
# 9. Aggregate and save results
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Saving aggregate metrics and artifacts")
print("=" * 80)

metrics_df = pd.DataFrame(all_metric_rows)
class_report_df = pd.concat(all_class_report_rows, ignore_index=True) if all_class_report_rows else pd.DataFrame()
predictions_df = pd.concat(all_prediction_frames, axis=0) if all_prediction_frames else pd.DataFrame()
probabilities_df = pd.concat(all_probability_frames, axis=0) if all_probability_frames else pd.DataFrame()
feature_importance_df = pd.concat(all_feature_importance_frames, ignore_index=True) if all_feature_importance_frames else pd.DataFrame()
model_summary_df = pd.DataFrame(run_model_summaries)

# Save metrics.
metrics_path_csv = METRIC_DIR / "model_zoo_metrics_all.csv"
metrics_path_parquet = METRIC_DIR / "model_zoo_metrics_all.parquet"
metrics_df.to_csv(metrics_path_csv, index=False)
metrics_df.to_parquet(metrics_path_parquet, index=False)

# Also save to global tables directory.
metrics_df.to_csv(TABLE_DIR / f"table_12_model_zoo_metrics_all_{RUN_ID}.csv", index=False)

# Save class-wise reports.
if not class_report_df.empty:
    class_report_df.to_csv(METRIC_DIR / "model_zoo_classification_reports_all.csv", index=False)
    class_report_df.to_parquet(METRIC_DIR / "model_zoo_classification_reports_all.parquet", index=False)
    class_report_df.to_csv(TABLE_DIR / f"table_13_model_zoo_classification_reports_all_{RUN_ID}.csv", index=False)

# Save aggregate predictions and probabilities.
if not predictions_df.empty:
    predictions_df.to_parquet(PRED_DIR / "predictions_all_models.parquet")
    predictions_df.to_csv(PRED_DIR / "predictions_all_models.csv")

if not probabilities_df.empty:
    probabilities_df.to_parquet(PROBA_DIR / "probabilities_all_models.parquet")
    probabilities_df.to_csv(PROBA_DIR / "probabilities_all_models.csv")

# Save feature importance.
if not feature_importance_df.empty:
    feature_importance_df.to_csv(IMPORTANCE_DIR / "feature_importance_all_models.csv", index=False)
    feature_importance_df.to_parquet(IMPORTANCE_DIR / "feature_importance_all_models.parquet", index=False)
    feature_importance_df.to_csv(TABLE_DIR / f"table_14_feature_importance_all_models_{RUN_ID}.csv", index=False)

# Save model summary.
model_summary_df.to_csv(METRIC_DIR / "model_training_summary.csv", index=False)

# ============================================================
# 10. Leaderboards
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Creating leaderboards")
print("=" * 80)

leaderboards = []

for target_col in TARGET_COLS:
    for split_name in ["validation", "test"]:
        tmp = metrics_df[
            (metrics_df["target_col"] == target_col)
            & (metrics_df["split"] == split_name)
        ].copy()

        if tmp.empty:
            continue

        # Primary baseline ranking:
        # high macro-F1, high balanced accuracy, low ordinal MAE.
        tmp["rank_macro_f1"] = tmp["macro_f1"].rank(ascending=False, method="min")
        tmp["rank_balanced_accuracy"] = tmp["balanced_accuracy"].rank(ascending=False, method="min")
        tmp["rank_ordinal_mae"] = tmp["ordinal_mae"].rank(ascending=True, method="min")
        tmp["composite_rank"] = (
            tmp["rank_macro_f1"]
            + tmp["rank_balanced_accuracy"]
            + tmp["rank_ordinal_mae"]
        ) / 3.0

        tmp = tmp.sort_values(
            ["composite_rank", "macro_f1", "balanced_accuracy", "ordinal_mae"],
            ascending=[True, False, False, True],
        )

        tmp.insert(0, "leaderboard_scope", f"{target_col}_{split_name}")
        leaderboards.append(tmp)

        print(f"\nLeaderboard: {target_col} | {split_name}")
        display_cols = [
            "model_name",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "weighted_f1",
            "quadratic_weighted_kappa",
            "ordinal_mae",
            "adjacent_accuracy_tol_1",
            "multiclass_log_loss",
            "composite_rank",
        ]
        print(tmp[display_cols].to_string(index=False))

        tmp.to_csv(
            METRIC_DIR / f"leaderboard_{safe_name(target_col)}_{split_name}.csv",
            index=False,
        )

leaderboard_df = pd.concat(leaderboards, ignore_index=True) if leaderboards else pd.DataFrame()
if not leaderboard_df.empty:
    leaderboard_df.to_csv(METRIC_DIR / "leaderboards_all.csv", index=False)
    leaderboard_df.to_parquet(METRIC_DIR / "leaderboards_all.parquet", index=False)
    leaderboard_df.to_csv(TABLE_DIR / f"table_15_model_zoo_leaderboards_{RUN_ID}.csv", index=False)

# Plot metric comparisons for test split.
for target_col in TARGET_COLS:
    for metric_name in [
        "macro_f1",
        "balanced_accuracy",
        "quadratic_weighted_kappa",
        "adjacent_accuracy_tol_1",
    ]:
        plot_metric_comparison(
            metrics_df,
            target_col=target_col,
            split_name="test",
            metric_name=metric_name,
            path=PLOT_DIR / f"model_comparison_{safe_name(target_col)}_test_{metric_name}.png",
        )

    # For ordinal MAE, lower is better, but the function sorts high to low.
    # Create a custom plot sorted ascending.
    tmp = metrics_df[
        (metrics_df["target_col"] == target_col)
        & (metrics_df["split"] == "test")
    ].copy()
    if not tmp.empty:
        tmp = tmp.sort_values("ordinal_mae", ascending=True)

        plt.figure(figsize=(10, max(4, 0.45 * len(tmp))))
        sns.barplot(data=tmp, y="model_name", x="ordinal_mae", color="#8172B2")
        plt.title(f"{target_col} | test | ordinal_mae lower is better")
        plt.xlabel("ordinal_mae")
        plt.ylabel("Model")
        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"model_comparison_{safe_name(target_col)}_test_ordinal_mae.png", dpi=200)
        plt.close()

# ============================================================
# 11. Save validation report
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Validation report and manifest")
print("=" * 80)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "03_AURORA_model_zoo_baselines.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "input_modeling_dataset": str(MODEL_DATA_PATH),
    "input_dataset_shape": df.shape,
    "input_date_range": {
        "start": str(df.index.min().date()),
        "end": str(df.index.max().date()),
    },
    "n_features": len(feature_cols),
    "target_cols": TARGET_COLS,
    "class_labels": CLASS_LABELS,
    "regime_label_definition": REGIME_LABEL_DEFINITION,
    "split_config": {
        "train_frac": TRAIN_FRAC,
        "validation_frac": VAL_FRAC,
        "test_frac": TEST_FRAC,
    },
    "split_report": split_report.to_dict(orient="records"),
    "model_config": MODEL_CONFIG,
    "n_metric_rows": int(len(metrics_df)),
    "n_prediction_rows": int(len(predictions_df)) if not predictions_df.empty else 0,
    "n_probability_rows": int(len(probabilities_df)) if not probabilities_df.empty else 0,
    "n_feature_importance_rows": int(len(feature_importance_df)) if not feature_importance_df.empty else 0,
    "model_training_summary": model_summary_df.to_dict(orient="records"),
    "primary_evaluation_note": (
        "For the paper, avoid relying only on accuracy. "
        "Use macro-F1, balanced accuracy, class-wise recall, confusion matrices, "
        "quadratic weighted kappa, ordinal MAE, and adjacent-class accuracy."
    ),
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "metrics": str(metrics_path_csv),
        "predictions_dir": str(PRED_DIR),
        "probabilities_dir": str(PROBA_DIR),
        "plots_dir": str(PLOT_DIR),
        "models_dir": str(ARTIFACT_MODEL_DIR),
        "feature_importance_dir": str(IMPORTANCE_DIR),
    },
}

save_json(REPORT_DIR / f"AURORA_03_model_zoo_validation_report_{RUN_ID}.json", validation_report)
save_json(RUN_ROOT / "AURORA_03_model_zoo_validation_report.json", validation_report)

manifest_df = make_file_manifest(RUN_ROOT)
manifest_df.to_csv(RUN_ROOT / "AURORA_03_model_zoo_file_manifest_SHA256.csv", index=False)
manifest_df.to_csv(REPORT_DIR / f"AURORA_03_model_zoo_file_manifest_SHA256_{RUN_ID}.csv", index=False)

# ============================================================
# 12. Final console summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA-TWETF MODEL ZOO BASELINES COMPLETE")
print("=" * 80)
print("Timestamp UTC:", RUN_TIMESTAMP)
print("Run ID       :", RUN_ID)
print("Run root     :", RUN_ROOT)
print("Metrics      :", metrics_path_csv)
print("Predictions  :", PRED_DIR)
print("Probabilities:", PROBA_DIR)
print("Plots        :", PLOT_DIR)
print("Models       :", ARTIFACT_MODEL_DIR)
print("Manifest     :", RUN_ROOT / "AURORA_03_model_zoo_file_manifest_SHA256.csv")
print("=" * 80)

print("\nFinal model training summary:")
print(model_summary_df.to_string(index=False))

print("\nSaved key tables:")
print(TABLE_DIR / f"table_10_model_zoo_split_report_{RUN_ID}.csv")
print(TABLE_DIR / f"table_12_model_zoo_metrics_all_{RUN_ID}.csv")
print(TABLE_DIR / f"table_13_model_zoo_classification_reports_all_{RUN_ID}.csv")
print(TABLE_DIR / f"table_15_model_zoo_leaderboards_{RUN_ID}.csv")

print("\nNext notebook:")
print("04_AURORA_ordinal_imbalance_uncertainty_models.ipynb")

Mounted at /content/drive
AURORA-TWETF Model Zoo Baselines
Timestamp UTC: 2026-06-23T14:57:28Z
Run ID       : 20260623_145728
Project root : /content/drive/MyDrive/AURORA_TWETF
Input data   : /content/drive/MyDrive/AURORA_TWETF/data/modeling/AURORA_TWETF_features_with_labels.parquet
Run root     : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/model_zoo_baselines/run_20260623_145728

Step 1: Loading modeling dataset
Dataset shape: (1262, 417)
Date range   : 2021-01-06 to 2026-03-25
Columns      : 417
Feature columns: 415
Target columns : ['TAIEX_regime_fixed_20d', 'TAIEX_regime_fixed_60d']

Step 2: Chronological train / validation / test split
     split   n start_date   end_date
     train 883 2021-01-06 2024-08-27
validation 189 2024-08-28 2025-06-12
      test 190 2025-06-13 2026-03-25

Step 3: Training and evaluating baseline model zoo

--------------------------------------------------------------------------------
Target: TAIEX_regime_fixed_20d
-------------------------